# ProNet – KNN Professional Connection Recommendation
This notebook demonstrates the ML module for recommending professional connections using K-Nearest Neighbors (KNN).

In [ ]:
# Install dependencies
# !pip install scikit-learn pandas numpy joblib matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import joblib
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully!')

## Step 1: Generate Dataset

In [ ]:
np.random.seed(42)
n_samples = 200

industries = ['Technology', 'Finance', 'Healthcare', 'Education', 'Marketing', 'Design', 'Data Science', 'Cybersecurity']
job_roles  = ['Software Engineer', 'Data Analyst', 'Product Manager', 'Designer', 'DevOps Engineer',
               'Data Scientist', 'Business Analyst', 'ML Engineer', 'Frontend Developer', 'Backend Developer']
skill_sets = {
    'Technology':    ['Python','Java','React','Node.js','Docker','AWS'],
    'Data Science':  ['Python','Machine Learning','SQL','TensorFlow','Statistics','R'],
    'Finance':       ['Excel','SQL','Python','Tableau','Financial Modeling','Bloomberg'],
    'Healthcare':    ['Clinical Research','Python','SPSS','SQL','Health Informatics','R'],
    'Education':     ['Curriculum Design','LMS','Python','Communication','Data Analysis'],
    'Marketing':     ['SEO','Google Analytics','Content Marketing','Social Media','HubSpot'],
    'Design':        ['Figma','Adobe XD','Sketch','Illustrator','CSS','HTML'],
    'Cybersecurity': ['Network Security','Python','Penetration Testing','SIEM','Kali Linux']
}
interest_topics = ['AI','Cloud Computing','Startups','Blockchain','Leadership','Open Source','FinTech','EdTech']

data = []
for i in range(n_samples):
    industry = np.random.choice(industries)
    role     = np.random.choice(job_roles)
    skills   = np.random.choice(skill_sets[industry], size=np.random.randint(2,5), replace=False).tolist()
    exp      = np.random.randint(0, 20)
    interests= np.random.choice(interest_topics, size=np.random.randint(1,3), replace=False).tolist()
    data.append({
        'id': f'user_{i+1}', 'name': f'Professional_{i+1}',
        'industry': industry, 'job_role': role,
        'skills': ', '.join(skills), 'primary_skill': skills[0],
        'experience_years': exp,
        'experience_level': 'Entry' if exp < 3 else ('Mid' if exp < 8 else 'Senior'),
        'interests': ', '.join(interests), 'primary_interest': interests[0]
    })

df = pd.DataFrame(data)
df.to_csv('dataset/professional_profiles.csv', index=False)
print(f'Dataset created: {df.shape}')
df.head()

## Step 2: Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['industry'].value_counts().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Profiles by Industry'); axes[0].set_xlabel('Industry'); axes[0].tick_params(axis='x', rotation=45)
df['experience_level'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Experience Levels')
plt.tight_layout(); plt.show()

## Step 3: Preprocessing & Encoding

In [ ]:
df_enc = df.copy()
categorical_cols = ['industry','job_role','experience_level','primary_skill','primary_interest']
encoders = {}
for col in categorical_cols:
    encoders[col] = LabelEncoder()
    df_enc[col+'_enc'] = encoders[col].fit_transform(df_enc[col])

feature_cols = [c+'_enc' for c in categorical_cols] + ['experience_years']
X = df_enc[feature_cols].values
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
print(f'Feature matrix: {X_scaled.shape}')
pd.DataFrame(X_scaled, columns=feature_cols).head()

## Step 4: Train KNN Model

In [ ]:
knn = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='brute')
knn.fit(X_scaled)
print('KNN model trained!')

## Step 5: Evaluation

In [ ]:
correct = 0; total = 50
for _ in range(total):
    idx = np.random.randint(0, len(X_scaled))
    d, inds = knn.kneighbors(X_scaled[idx].reshape(1,-1), n_neighbors=6)
    neighbors = inds[0][1:]
    target_ind = df.iloc[idx]['industry']
    neighbor_ind = df.iloc[neighbors]['industry'].values
    if np.sum(neighbor_ind == target_ind) >= 2:
        correct += 1
print(f'Same-industry Precision@5: {correct/total:.2%}')

## Step 6: Demo Recommendations

In [ ]:
test_user = {'industry':'Technology','job_role':'Software Engineer',
             'experience_level':'Mid','primary_skill':'Python',
             'primary_interest':'AI','experience_years':4}

test_feats = [encoders[c].transform([test_user[c]])[0] if test_user[c] in encoders[c].classes_ else 0
              for c in categorical_cols] + [test_user['experience_years']]
test_scaled = scaler.transform([test_feats])
dists, inds = knn.kneighbors(test_scaled, n_neighbors=6)

print('Top 5 Recommended Connections:')
for i, idx in enumerate(inds[0][1:]):
    c = df.iloc[idx]
    score = round((1 - dists[0][i+1]) * 100, 1)
    print(f'{i+1}. {c["name"]} | {c["job_role"]} | {c["industry"]} | Match: {score}%')

## Step 7: Save Model

In [ ]:
import os; os.makedirs('model', exist_ok=True)
joblib.dump({'model':knn,'scaler':scaler,'encoders':encoders,
             'feature_cols':feature_cols,'encoded_data':X_scaled,'original_data':df,'k':5},
            'model/knn_model.pkl')
print('Model saved to model/knn_model.pkl')